In [ ]:
%reload_ext autoreload
%autoreload 2 
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

viz.apply_theme(theme="default")

con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

In [ ]:
raw = pd.read_sql("SELECT * FROM ods_signal", con)

raw['ts'] = pd.to_datetime(raw['timestamp'], unit="ms") + \
    pd.Timedelta(hours = config.UTC_OFFSET_HOURS)
raw.shape

In [ ]:
missing = pd.DataFrame({
    "缺失数":
    raw.drop(columns='ts').isna().sum() + raw.drop(columns="ts").apply(
        lambda x: x.astype('string').str.strip().eq("").sum() if
        (pd.api.types.is_string_dtype(x) or x.dtype == object) else 0)
}).rename_axis('字段').reset_index()
missing['缺失率'] = round(missing['缺失数'] / len(raw),7)
missing = missing.sort_values(by='缺失率').reset_index(drop=True)
missing.to_sql("ads_missing_by_field", con, if_exists="replace", index=False)
missing

In [ ]:
m = pd.read_sql("SELECT * FROM ads_missing_by_field", con)
full = m[m['缺失率'] == 1]  # 正列为空的字段
worst_other = m.loc[m['缺失率'] < 1, '缺失数'].max()

fig, ax = plt.subplots(figsize=(10, 5.6), dpi=100)
colors = [RED if r == 1 else GRAY for r in m['缺失率']]
bars = ax.barh(m['字段'], m['缺失率'], color=colors, height=.62)
labels = [
    '100%(整列为空)' if r == 1 else (f"{n}行" if n > 0 else "0")
    for r, n in zip(m['缺失率'], m['缺失数'])
]

ax.bar_label(bars, labels = labels, padding = 4, color = INK)
ax.set_xlim(0, 1.25)
ax.grid()
title = f"{', '.join(sorted(full['字段']))} 整列为空，其余字段缺失最多为{worst_other}"
viz.finish(fig, title, 
    f"各字段缺失率：空值和空白字符串都算缺失，口径：ods_signal全部{len(raw):,}行",
    SOURCE,"D3_01_字段缺失率.png"
    )

In [ ]:
quality.dq_record(con, "DQ_02","完整性",
    "整列为空的字段","ods_signal.tmp0 / tmp1", "2个字段 100%为空",'不通过',
    evidence="tmp0 全部为空值，tmp1全部为空格", decision="删除这两个字段"
    )

quality.dq_record(con, "DQ_03","完整性",
    "关键字段缺失","ods_signal.imsi / lac_id / cell_id",
    f"{raw[['imsi', 'lac_id','cell_id']].isna().any(axis=1).sum()}行","通过",
    evidence="imsi缺15行，lac_id缺4, cell_id缺5，占比不到0.01%",
    decision="删除这些行"
    )

In [ ]:
ids = raw['imsi'].dropna()
bad_imsi = raw[raw['imsi'].notna()
    & ~raw['imsi'].str.fullmatch(r"\d{18}", na=False)]
print(f"IMSI非空 {len(ids):,}行，其中长度为18的 {ids.str.len().eq(18).sum():,}行")
print(f"不是 18位纯数字的: {len(bad_imsi)}行")
bad_imsi[['imsi', 'phone', 'lac_id', 'cell_id', 'nid']]

In [ ]:
phone_len = raw['phone'].str.len().value_counts()
odd_phone = raw.loc[raw['phone'].str.len() != 14, 'phone'].value_counts()
print(f"手机号长度分布: {phone_len.to_dict()}")
print(f'异常手机号: {odd_phone.to_dict()}, \
涉及 {raw.loc[raw["phone"] == "6666",'imsi'].nunique()}个 IMSI')

In [ ]:
quality.dq_record(con, "DQ-04", "规范性", "IMSI 格式", "ods_signal.imsi", f"{len(bad_imsi)} 行含 * ^ # 等掩码字符", "警告",
          evidence="长度都是 18，但不是纯数字，疑似脱敏时误处理", decision="删除这些行，并反馈数据提供方")
quality.dq_record(con, "DQ-05", "规范性", "手机号格式", "ods_signal.phone", "36 行为占位值 6666（14 个 IMSI）", "警告",
          evidence="其余手机号均为 14 位", decision="手机号不作为主键使用；分析中不需要手机号，明细表不保留")

In [ ]:
raw.columns

In [ ]:
orig_cols = ['timestamp', 'imsi', 'lac_id', 'cell_id',
             'phone', 'timestamp1','nid', 'npid']

# 这个是根据业务作出决定，因为关键字段缺失了就没法定位，所以整行丢掉
step = raw.dropna(subset=['imsi','lac_id','cell_id'])
# 格式不对，比如IMSI有问题，就定位不到人了
step = step[step['imsi'].str.fullmatch(r"\d{18}")]
# 每组重复只保留第一条
dup_mask = step.duplicated(subset=orig_cols, keep="first")
print(f'完全重复: {dup_mask.sum():,}行,占比{dup_mask.mean():.2%}')

step[step.duplicated(subset=orig_cols,keep=False)].sort_values(orig_cols).head(5)

In [ ]:
orig_cols = ['timestamp', 'imsi', 'lac_id', 'cell_id',
            'phone', 'timestamp1','nid', 'npid']

# 这个是根据业务作出决定，因为关键字段缺失了就没法定位，所以整行丢掉
step = raw.dropna(subset=['imsi','lac_id','cell_id'])
# 格式不对，比如IMSI有问题，就定位不到人了
step = step[step['imsi'].str.fullmatch(r"\d{18}")]
# 每组重复只保留第一条
dup_mask = step.duplicated(subset=orig_cols, keep="first")
print(f'完全重复：{dup_mask.sum():,}行,占比{dup_mask.mean():.2%}')

step[step.duplicated(subset=orig_cols,keep=False)].sort_values(orig_cols).head(5)

In [ ]:
valid = raw.dropna(subset=['imsi'])
valid = valid[valid['imsi'].str.fullmatch(r"\d{18}")
              & (valid['phone'] != "6666")]
# 只看那个imsi 配过那个手机号，这组对应关系，同一对出现多少次不重要，所以要先去重
pairs = valid[['imsi','phone']].drop_duplicates()
# 裁剪出前15位，作为另外一种认证方式
pairs['imsi15']  = pairs['imsi'].str[:15]

per18 = pairs.groupby("imsi")['phone'].nunique()
per15 = pairs.groupby('imsi15')['phone'].nunique()

key_check = pd.DataFrame({
    "分组方式": ['按前15位',"按完整18位"],
    "ID个数": [per15.shape[0], per18.shape[0]],
    "对应多个手机号的ID": [(per15 > 1).sum(), (per18 > 1).sum()]
})
key_check['占比'] = (key_check['对应多个手机号的ID'] / key_check['ID个数']).round(3)
key_check.to_sql("ads_key_check", con, if_exists="replace",index=False)

In [ ]:
multi = per15[per15 > 1].index
affected = pairs.loc[pairs['imsi15'].isin(multi),'imsi'].nunique()
suffuix_kinds = pairs['imsi'].str[-3:].nunique()
print(f"对应多个手机号的前15位分组："
    f"{len(multi)}组，里面装着{affected:,}个不同的18位IMSI,"
    f"占全部用户的{affected / len(per18):.1%}")
print(f"后三位一共有{suffuix_kinds}种取值")

In [ ]:
k = pd.read_sql("SELECT * FROM ads_key_check", con)
rows = [(f"按前 15 位分组\n（{k.loc[0, 'ID个数']:,} 组）", 
         k.loc[0, "对应多个手机号的ID"],
         k.loc[0, "ID个数"], "组对应多个手机号"),
        (f"被牵连的用户\n（{len(per18):,} 个 18 位 IMSI）", affected, len(per18),
         "个用户会被错误合并"),
        (f"按完整 18 位\n（{k.loc[1, 'ID个数']:,} 个）", k.loc[1, "对应多个手机号的ID"],
         k.loc[1, "ID个数"], "个对应多个手机号")]
rows

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# ---- 数据准备 ----
# rows 结构: (label, num, den, note)
labels = [r[0] for r in rows]
nums   = [int(r[1]) for r in rows]
dens   = [int(r[2]) for r in rows]
notes  = [r[3] for r in rows]
ratios = [n / d for n, d in zip(nums, dens)]

# 语义色：第1、2行是"问题"(红)，第3行是"健康"(主色)
bar_colors = [RED, RED, MAIN]
# 背景槽（表示分母总量）用极浅灰
track_color = FAINT

# ---- 绘图 ----
fig, ax = plt.subplots(figsize=(10, 3.6))
y = range(len(rows))[::-1]   # 从上往下：① ② ③

BAR_H = 0.42

for yi, ratio, num, den, color in zip(y, ratios, nums, dens, bar_colors):
    # 1) 底槽：整条代表分母
    ax.barh(yi, 1.0, height=BAR_H, color=track_color, zorder=1)
    # 2) 实条：分子/分母
    ax.barh(yi, ratio, height=BAR_H, color=color, zorder=2)
    # 3) 数值标签（柱右端外侧）
    ax.text(max(ratio, 0.02) + 0.015, yi,
            f"{num:,} / {den:,}  ({ratio:.1%})",
            va="center", ha="left",
            color=INK, fontsize=10, fontweight="bold")

# ---- 轴与网格 ----
ax.set_yticks(list(y))
ax.set_yticklabels(labels, color=INK, fontsize=10)
ax.set_xlim(0, 1.30)          # 留出右侧标数空间
ax.set_xticks([])             # 隐藏 X 刻度，比例已写在标签里
for s in ("top", "right", "bottom", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(False)

# ---- 标题：主结论 + 副依据 ----
ax.set_title(
    "用户主键从 15 位 IMSI 切换到完整 18 位后，手机号冲突归零",
    color=INK, fontsize=13, fontweight="bold", loc="left", pad=14
)
ax.text(0, 1.04, "对比口径：同一分组下的手机号个数",
        transform=ax.transAxes, color=MUTED, fontsize=9, ha="left")

# ---- 图例（3 个色块，替代刻度说明）----
from matplotlib.patches import Patch
handles = [
    Patch(facecolor=RED,  label="存在问题（会误合并用户）"),
    Patch(facecolor=MAIN, label="推荐方案（唯一）"),
]
ax.legend(handles=handles, loc="lower right",
          frameon=False, fontsize=9, labelcolor=MUTED)

# ---- 底部来源标注 ----
fig.text(0.01, -0.02, SOURCE, color=FAINT, fontsize=8, ha="left")

plt.tight_layout()
plt.show()

In [ ]:
orig_cols = ['timestamp', 'imsi', 'lac_id', 'cell_id',
            'phone', 'timestamp1','nid', 'npid']
steps = []
df = raw.drop(columns=['tmp0','tmp1'])
n0 = len(df)

def log_step(name, before, after):
    steps.append({"步骤": name, "剔除行数":before - after,"剩余行数":after})
    
n = len(df); df = df.dropna(subset=["imsi", "lac_id", "cell_id"])
log_step("缺 imsi / lac / cell", n, len(df))

n = len(df); df = df[df['imsi'].str.fullmatch(r"\d{18}")]
log_step('IMSI 格式异常', n, len(df))

n = len(df); df = df.drop_duplicates(subset=orig_cols)
log_step("完全重复", n, len(df))

n = len(df); df = df[df['ts'].dt.strftime("%Y-%m-%d") == DAY]
log_step("不是10月3日", n, len(df))

steps = pd.DataFrame(steps)
steps['剔除占比'] = steps['剔除行数'] / n0
steps.to_sql("ads_clean_steps", con, if_exists="replace", index=False)
steps

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

# ---------- 数据准备 ----------
# steps 已含"步骤/剔除行数/剩余行数/剔除占比"
# 补一个起点：原始总行数 n0
N0 = int(n0)                        # 你脚本里的 n0
final_n = int(steps['剩余行数'].iloc[-1])

# 组装绘图数据：起点 + 4 步 + 终点
plot_df = pd.concat([
    pd.DataFrame([{"步骤": f"原始数据", "剔除行数": 0, "剩余行数": N0, "剔除占比": 0.0}]),
    steps[["步骤", "剔除行数", "剩余行数", "剔除占比"]],
], ignore_index=True)

labels   = plot_df["步骤"].tolist()
remain   = plot_df["剩余行数"].tolist()
cut      = plot_df["剔除行数"].tolist()
cut_pct  = plot_df["剔除占比"].tolist()

y = list(range(len(labels)))[::-1]      # 从上往下
BAR_H = 0.55

# ---------- 绘图 ----------
fig, ax = plt.subplots(figsize=(11, 4.6))

for i, yi in enumerate(y):
    # 底槽：整条 = N0（表示"原始规模"作为参照系）
    ax.barh(yi, N0, height=BAR_H, color=FAINT, zorder=1)

    # 剩余条：颜色区分 —— 起点/终点用 MAIN，中间步骤仍用 MAIN（保留量）
    color = MAIN if i == 0 or i == len(y) - 1 else MAIN_MID
    ax.barh(yi, remain[i], height=BAR_H, color=color, zorder=2)

    # 剩余量数值（柱内右侧）
    ax.text(remain[i] - N0 * 0.01, yi, f"{remain[i]:,}",
            va="center", ha="right",
            color="white", fontsize=10, fontweight="bold", zorder=3)

    # 剔除量 + 占比（柱右端外侧）—— 起点无剔除，跳过
    if cut[i] > 0:
        ax.text(remain[i] + N0 * 0.012, yi,
                f"−{cut[i]:,}  ({cut_pct[i]:.1%})",
                va="center", ha="left",
                color=RED, fontsize=10, fontweight="bold", zorder=3)
    else:
        ax.text(remain[i] + N0 * 0.012, yi,
                f"= 基线",
                va="center", ha="left",
                color=MUTED, fontsize=10, zorder=3)

# ---------- 轴 ----------
ax.set_yticks(y)
ax.set_yticklabels(labels, color=INK, fontsize=10.5)
ax.set_xlim(0, N0 * 1.20)     # 右侧留白放剔除标注
ax.set_xticks([])
for s in ("top", "right", "bottom", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(False)

# ---------- 标题 ----------
ax.set_title("数据清洗四步漏斗：共剔除 {} 行，最终保留 {:,} 行".format(
                f"{int(steps['剔除行数'].sum()):,}", final_n),
             color=INK, fontsize=13, fontweight="bold", loc="left", pad=16)
ax.text(0, 1.045,
        f"原始 {N0:,} 行 → 最终 {final_n:,} 行，保留率 {final_n/N0:.1%}",
        transform=ax.transAxes, color=MUTED, fontsize=9.5, ha="left")

# ---------- 图例 ----------
handles = [
    Patch(facecolor=MAIN,     label="剩余可用数据"),
    Patch(facecolor=FAINT,    label="原始规模（参照）"),
    Patch(facecolor="white", edgecolor=RED, label="本步剔除量"),
]
ax.legend(handles=handles, loc="lower right",
          frameon=False, fontsize=9, labelcolor=MUTED)

# ---------- 来源标注 ----------
fig.text(0.01, -0.02, SOURCE, color=FAINT, fontsize=8, ha="left")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

# ---------- 数据准备 ----------
N0      = int(n0)                       # 532828
final_n = int(steps["剩余行数"].iloc[-1])  # 515931

plot_df = pd.concat([
    pd.DataFrame([{"步骤": "原始数据", "剔除行数": 0,
                   "剩余行数": N0, "剔除占比": 0.0}]),
    steps[["步骤", "剔除行数", "剩余行数", "剔除占比"]],
], ignore_index=True)

labels  = plot_df["步骤"].tolist()
remain  = plot_df["剩余行数"].tolist()
cut     = plot_df["剔除行数"].tolist()
cut_pct = plot_df["剔除占比"].tolist()

y = list(range(len(labels)))[::-1]
BAR_H = 0.55
XMAX  = N0 * 1.34          # 右侧留足空间，避免标数溢出

# ---------- 绘图 ----------
fig, ax = plt.subplots(figsize=(11.5, 4.8))

for i, yi in enumerate(y):
    # 底槽（原始规模参照）
    ax.barh(yi, N0, height=BAR_H, color=FAINT, zorder=1)

    # 剩余条：起点/终点深色，中间步骤浅色
    if i == 0 or i == len(y) - 1:
        color = MAIN_DARK
    else:
        color = MAIN_LIGHT
    ax.barh(yi, remain[i], height=BAR_H, color=color, zorder=2)

    # 剩余量：写在柱内部靠右
    ax.text(remain[i] - N0 * 0.015, yi, f"{remain[i]:,}",
            va="center", ha="right",
            color="white", fontsize=10, fontweight="bold", zorder=3)

    # 剔除量：写在固定右侧列（关键：位置固定，杜绝与柱尾重叠）
    tag_x = N0 * 1.02                      # 固定起始 x
    if cut[i] > 0:
        ax.text(tag_x, yi,
                f"−{cut[i]:,}   ({cut_pct[i]:.1%})",
                va="center", ha="left",
                color=RED, fontsize=10, fontweight="bold", zorder=3)
    else:
        ax.text(tag_x, yi, "= 基线",
                va="center", ha="left",
                color=MUTED, fontsize=10, zorder=3)

# ---------- 轴 ----------
ax.set_yticks(y)
ax.set_yticklabels(labels, color=INK, fontsize=10.5)
ax.set_xlim(0, XMAX)
ax.set_xticks([])
for s in ("top", "right", "bottom", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(False)

# ---------- 标题：仅一行，不再堆副标题 ----------
ax.set_title(
    f"数据清洗四步漏斗：共剔除 {int(steps['剔除行数'].sum()):,} 行，最终保留 {final_n:,} 行",
    color=INK, fontsize=13, fontweight="bold", loc="left", pad=18
)

# ---------- 图例：放到图外右上，彻底不侵占数据 ----------
handles = [
    Patch(facecolor=MAIN_DARK,  label="原始 / 最终"),
    Patch(facecolor=MAIN_LIGHT, label="中间步骤剩余"),
    Patch(facecolor=FAINT,      label="原始规模（参照）"),
    Patch(facecolor="white", edgecolor=RED, label="本步剔除量"),
]
ax.legend(handles=handles,
          loc="lower left", bbox_to_anchor=(0.0, 1.02),   # 图外、标题下方
          ncol=4, frameon=False, fontsize=9,
          labelcolor=MUTED, handlelength=1.4, columnspacing=1.6)

# ---------- 保留率：放左下角（与原副标题同义，不挤标题） ----------
ax.text(0.0, -0.12,
        f"保留率 {final_n / N0:.1%}（{N0:,} → {final_n:,}）",
        transform=ax.transAxes, color=MUTED, fontsize=9.5, ha="left")

# ---------- 来源标注 ----------
fig.text(0.01, -0.06, SOURCE, color=FAINT, fontsize=8, ha="left")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.9))

# ...（绘制柱、标数、轴设置部分不变）...

# ---------- 图例：贴在轴顶上方，一行横排 ----------
handles = [
    Patch(facecolor=MAIN_DARK,  label="原始 / 最终"),
    Patch(facecolor=MAIN_LIGHT, label="中间步骤剩余"),
    Patch(facecolor=FAINT,      label="原始规模（参照）"),
    Patch(facecolor="white", edgecolor=RED, label="本步剔除量"),
]
ax.legend(handles=handles,
          loc="lower left", bbox_to_anchor=(0.0, 1.01),
          ncol=4, frameon=False, fontsize=9,
          labelcolor=MUTED, handlelength=1.4, columnspacing=1.6)

# ---------- 标题：改用 fig.suptitle，位置在图最顶 ----------
fig.suptitle(
    f"数据清洗四步漏斗：共剔除 {int(steps['剔除行数'].sum()):,} 行，最终保留 {final_n:,} 行",
    color=INK, fontsize=13, fontweight="bold",
    x=0.012, y=1.02, ha="left", va="bottom"
)

# ---------- 保留率：图左下角 ----------
ax.text(0.0, -0.12,
        f"保留率 {final_n / N0:.1%}（{N0:,} → {final_n:,}）",
        transform=ax.transAxes, color=MUTED, fontsize=9.5, ha="left")

fig.text(0.01, -0.06, SOURCE, color=FAINT, fontsize=8, ha="left")

# 关键：给标题/图例腾出顶部空间
plt.subplots_adjust(top=0.80, bottom=0.14, left=0.16, right=0.97)
plt.show()

In [ ]:
# ============================================================
# 数据清洗四步漏斗可视化
# 依赖：pandas, matplotlib
# 前置变量：raw, n0, steps, DAY, SOURCE（来自你的分析脚本）
# 配色常量：MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY,
#           GRAY_DARK, RED, AMBER, INK, MUTED, FAINT
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# ------------------------------------------------------------
# 1. 数据准备
# ------------------------------------------------------------
N0      = int(n0)                             # 原始总行数
final_n = int(steps["剩余行数"].iloc[-1])     # 最终保留行数
total_cut = int(steps["剔除行数"].sum())      # 总剔除行数

# 在最前面插入一行"原始数据"作为基线
plot_df = pd.concat([
    pd.DataFrame([{
        "步骤": "原始数据",
        "剔除行数": 0,
        "剩余行数": N0,
        "剔除占比": 0.0,
    }]),
    steps[["步骤", "剔除行数", "剩余行数", "剔除占比"]],
], ignore_index=True)

labels  = plot_df["步骤"].tolist()
remain  = plot_df["剩余行数"].tolist()
cut     = plot_df["剔除行数"].tolist()
cut_pct = plot_df["剔除占比"].tolist()

# 纵向位置：从上往下排（原始数据在最上）
y     = list(range(len(labels)))[::-1]
BAR_H = 0.55
XMAX  = N0 * 1.34          # 右侧留白，容纳剔除量标注


# ------------------------------------------------------------
# 2. 绘图
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11.5, 5.4))

for i, yi in enumerate(y):
    # 2.1 底槽：代表原始规模，作为参照系
    ax.barh(yi, N0, height=BAR_H, color=FAINT, zorder=1)

    # 2.2 剩余条：起点/终点用深色，中间步骤用浅色
    color = MAIN_DARK if i in (0, len(y) - 1) else MAIN_LIGHT
    ax.barh(yi, remain[i], height=BAR_H, color=color, zorder=2)

    # 2.3 剩余量数值：写在柱内部靠右，白色反白
    ax.text(
        remain[i] - N0 * 0.015, yi,
        f"{remain[i]:,}",
        va="center", ha="right",
        color="white", fontsize=10, fontweight="bold", zorder=3,
    )

    # 2.4 剔除量：固定写在右侧列（不与柱尾浮动，避免重叠）
    tag_x = N0 * 1.02
    if cut[i] > 0:
        ax.text(
            tag_x, yi,
            f"−{cut[i]:,}   ({cut_pct[i]:.1%})",
            va="center", ha="left",
            color=RED, fontsize=10, fontweight="bold", zorder=3,
        )
    else:
        ax.text(
            tag_x, yi,
            "= 基线",
            va="center", ha="left",
            color=MUTED, fontsize=10, zorder=3,
        )


# ------------------------------------------------------------
# 3. 坐标轴清理
# ------------------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(labels, color=INK, fontsize=10.5)
ax.set_xlim(0, XMAX)
ax.set_xticks([])

for s in ("top", "right", "bottom", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(False)


# ------------------------------------------------------------
# 4. 标题（轴顶，pad=20 与下方内容隔开）
# ------------------------------------------------------------
ax.set_title(
    f"数据清洗四步漏斗：共剔除 {total_cut:,} 行，最终保留 {final_n:,} 行",
    color=INK, fontsize=13, fontweight="bold",
    loc="left", pad=20,
)


# ------------------------------------------------------------
# 5. 图例（沉底，位于轴下方，绝不与标题重叠）
# ------------------------------------------------------------
handles = [
    Patch(facecolor=MAIN_DARK,               label="原始 / 最终"),
    Patch(facecolor=MAIN_LIGHT,              label="中间步骤剩余"),
    Patch(facecolor=FAINT,                   label="原始规模（参照）"),
    Patch(facecolor="white", edgecolor=RED,  label="本步剔除量"),
]
ax.legend(
    handles=handles,
    loc="upper left", bbox_to_anchor=(0.0, -0.12),
    ncol=4, frameon=False, fontsize=9,
    labelcolor=MUTED, handlelength=1.4, columnspacing=1.6,
)


# ------------------------------------------------------------
# 6. 保留率 + 来源标注（图最底部）
# ------------------------------------------------------------
ax.text(
    0.0, -0.26,
    f"保留率 {final_n / N0:.1%}（{N0:,} → {final_n:,}）",
    transform=ax.transAxes,
    color=MUTED, fontsize=9.5, ha="left",
)
fig.text(
    0.01, -0.10,
    SOURCE,
    color=FAINT, fontsize=8, ha="left",
)


# ------------------------------------------------------------
# 7. 边距微调（给标题、图例、来源留出空间）
# ------------------------------------------------------------
plt.subplots_adjust(top=0.90, bottom=0.24, left=0.16, right=0.97)
plt.show()

In [ ]:
dwd = pd.DataFrame({
    "imsi": df['imsi'],
    "ts": df['ts'],
    "dt": DAY,
    "lac_id": df['lac_id'].astype("int64"),
    "cell_id":df['cell_id'].astype("int64"),
    "nid": df['nid']
})

dwd["lac_cell"] = dwd['lac_id'].astype(str) + "-" + dwd['cell_id'].astype(str)

dwd = dwd.sort_values(['imsi','ts']).reset_index(drop=True)

dwd['ts'] = dwd['ts'].dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

dwd.to_sql("dwd_signal", con, if_exists='replace', index=False)

In [ ]:
x = dwd[['imsi','ts','lac_id','nid']].copy()
prev_lac = x.groupby("imsi")['lac_id'].shift()

x["LAC变化"] = prev_lac.notna() & (x['lac_id'] != prev_lac)
x = x[prev_lac.notna()]
by_nid = x.groupby('nid').agg(
    记录数= ("LAC变化", "size"),
    LAC变化占比=("LAC变化", "mean")
)

top12 = by_nid.nlargest(12, '记录数').sort_values("LAC变化占比")
overall = x['LAC变化'].mean()
top12.to_sql("ads_nid_lac_change", con, if_exists='replace')

In [ ]:
# ============================================================
# D05_03 · nid 位置区（LAC）变化可视化
# 依赖：pandas, matplotlib, signal_insight.viz / config
# ============================================================

import matplotlib.pyplot as plt
from signal_insight import viz, config

HIGH = 0.70  # "明显伴随位置区变化"的阈值

# ---------- 1. 数据准备（你原逻辑，加 .values 更稳） ----------
d       = top12.sort_values("LAC变化占比")
r       = d["LAC变化占比"].values
n_rec   = d["记录数"].values
labels  = [f"nid {n}" for n in d.index]

# 三档配色
colors = [
    viz.MAIN_DARK  if v >= HIGH    else
    viz.MAIN_LIGHT if v >  overall else
    viz.GRAY
    for v in r
]

# ---------- 2. 绘图 ----------
fig, ax = plt.subplots(figsize=(10.5, 5.8), dpi=110)
y = list(range(len(d)))

ax.barh(y, r, color=colors, height=0.68, zorder=2)

# ---------- 3. 坐标轴 ----------
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10.5, color=viz.INK)
ax.set_xlim(0, 1.20)                    # 右侧留白，容两列标注
ax.set_ylim(-0.7, len(d) - 0.3)         # 顶部留白，容基准文字
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.tick_params(axis="x", labelsize=10, colors=viz.MUTED)
ax.tick_params(axis="y", length=0, colors=viz.INK)

for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(viz.GRAY)
ax.grid(False)

# ---------- 4. 条尾标注：两列固定对齐（避免溢出/重叠） ----------
X_RATIO = 0.92   # 占比列
X_COUNT = 1.04   # 记录数列
for i, (v, n) in enumerate(zip(r, n_rec)):
    ax.text(X_RATIO, i, f"{v:.0%}",
            va="center", ha="left",
            fontsize=10, color=viz.INK,
            fontweight="bold" if v >= HIGH else "normal",
            zorder=3)
    ax.text(X_COUNT, i, f"{n:,} 条",
            va="center", ha="left",
            fontsize=9, color=viz.MUTED,
            zorder=3)

# ---------- 5. 整体基准线 ----------
ax.axvline(overall, color=viz.MUTED, ls="--", lw=1, zorder=1)
ax.text(overall, len(d) - 0.25, f"全部记录 {overall:.0%}",
        ha="center", va="bottom",
        fontsize=9.5, color=viz.MUTED)

# ---------- 6. 标题 / 副标题（结论式，空集保护） ----------
hi = d[d["LAC变化占比"] >= HIGH].sort_values("LAC变化占比", ascending=False)
if len(hi) > 0:
    title = (f"nid {'、'.join(map(str, hi.index))} 伴随位置区变化 "
             f"{hi['LAC变化占比'].min():.0%}–{hi['LAC变化占比'].max():.0%}，"
             f"整体仅 {overall:.0%}")
else:
    title = f"无 nid 达到 {HIGH:.0%} 位置区变化阈值（整体 {overall:.0%}）"

subtitle = (
    f"记录数前 12 的 nid（共 {d['记录数'].sum():,} 条）；按用户、时间排序，"
    f"比较每条记录的 LAC 与本人上一条是否不同。\n"
    f"深蓝 ≥ {HIGH:.0%}，像位置更新类事件；灰色低于整体，不随位置区变化。"
    f"只说明相关，两条记录间可能隔很久。"
)

SOURCE = f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；制表：{config.AUTHOR}"

# ---------- 7. 收尾（统一主题 + 保存） ----------
viz.finish(fig, title, subtitle, SOURCE, "D05_03_nid位置区变化.png")
plt.show()

In [ ]:

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

In [ ]:

s = pd.read_sql("SELECT imsi, ts FROM dwd_signal",con, parse_dates="ts").sort_values(['imsi','ts'])

In [ ]:
obs = s.assign(h=s['ts'].dt.hour).groupby("h")['imsi'].nunique().reindex(
    range(24), fill_value=0).reset_index()
obs['占比'] = (obs['imsi'] /  s['imsi'].nunique()).round(2)

In [ ]:
from matplotlib.ticker import FixedLocator, FixedFormatter, PercentFormatter

# 来源行：脚本名、配置版本按实际填写（第 1 节要求：数据来源 + 脚本 + 配置版本）
SOURCE = f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；制图: {config.AUTHOR}"

def fmt_pct(p, digits=0):
    """第 6 节：一般取整；很小的值自动多保留位数（0.01%，不写 0%）。"""
    if p == 0:
        return "0%"
    while round(p * 100, digits) == 0 and digits < 4:
        digits += 1
    return f"{p:.{digits}%}"


def fmt_dur(minutes):
    """第 6 节：单位写全；超过 2 小时改用小时。"""
    if minutes < 1:
        return f"{minutes * 60:.0f} 秒"
    if minutes < 10:
        return f"{minutes:.1f} 分钟"
    if minutes <= 120:
        return f"{minutes:,.0f} 分钟"
    return f"{minutes / 60:.1f} 小时"


# %% 1 口径检查 + 分组计算
# 每小时人数按“小时”分组；数据跨多天时同一小时会把不同天的人混在一起，必须先按日期分组
n_days = s["ts"].dt.normalize().nunique()
assert n_days == 1, f"数据跨 {n_days} 天：请改为按 日期+小时 分组"
DAY_LABEL = s["ts"].dt.strftime("%m-%d").iloc[0]

# diff 依赖行顺序：先按 人、时间 排序，再在每个人内部做差，保证间隔不跨人、不出负数
s = s.sort_values(["imsi", "ts"])
gap = (s.groupby("imsi")["ts"].diff().dt.total_seconds() / 60).dropna()
assert (gap >= 0).all()

n_user = s["imsi"].nunique()
n_rec = len(s)
n_gap = len(gap)
p50, p90, p99 = gap.quantile([0.5, 0.9, 0.99])
share_zero = (gap == 0).mean()          # 同一时刻的重复记录
share_gt60 = (gap > 60).mean()          # 按“条数”算：超过 1 小时的间隔占比
time_gt60 = gap[gap > 60].sum() / gap.sum()  # 按“时间”算：这些长间隔覆盖的时长占比

hourly = (s.assign(h=s["ts"].dt.hour)
           .groupby("h")["imsi"].nunique()
           .reindex(range(24), fill_value=0)
           .rename("人数").to_frame())
hourly["占比"] = hourly["人数"] / n_user   # 不先四舍五入，展示时再格式化


# %% 2 图一：相邻信令间隔的累计分布（ECDF，对数横轴）
x_sorted = np.sort(gap.to_numpy())
x_min = 1 / 60                                   # 1 秒
grid = np.geomspace(x_min, 24 * 60, 600)
ecdf = np.searchsorted(x_sorted, grid, side="right") / n_gap   # 0 秒间隔计入曲线起点



In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grid, ecdf, color=MAIN, lw=2.2)
ax.set_xscale("log")
ax.set_xlim(x_min, 24 * 60)
ax.set_ylim(0, 1.02)

# 横轴标实际时长，不标 10^n
ticks = [1/60, 10/60, 1, 5, 15, 60, 6 * 60, 24 * 60]
labels = ["1 秒", "10 秒", "1 分钟", "5 分钟", "15 分钟", "1 小时", "6 小时", "24 小时"]
ax.xaxis.set_major_locator(FixedLocator(ticks))
ax.xaxis.set_major_formatter(FixedFormatter(labels))
ax.xaxis.set_minor_locator(FixedLocator([]))
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xlabel("同一人相邻两条信令的间隔（对数刻度）")
ax.set_ylabel("间隔不超过该值的占比")
ax.grid(axis="y", color=GRAY, lw=0.6, alpha=0.6)

# 1 小时参考线：和“每小时一个位置”的口径直接相关
ax.axvline(60, color=GRAY_DARK, lw=1, ls="--")
ax.text(60 * 1.15, 0.06, f"超过 1 小时的间隔\n按条数占 {fmt_pct(share_gt60, 1)}\n"
        f"按时长占 {fmt_pct(time_gt60)}", color=MUTED, fontsize=9.5, va="bottom")

# 三个分位点：打点 + 直接标注（不用图例）
for q, v, txt, dx, dy, ha, va in [
    (0.50, p50, f"一半的间隔 ≤ {fmt_dur(p50)}", 10, -4, "left", "top"),
    (0.90, p90, f"90% ≤ {fmt_dur(p90)}", -10, 4, "right", "bottom"),
    (0.99, p99, f"99% ≤ {fmt_dur(p99)}", 10, -6, "left", "top"),
]:
    ax.plot(v, q, "o", color=MAIN_DARK, ms=6, zorder=3)
    ax.annotate(txt, (v, q), xytext=(dx, dy), textcoords="offset points",
                ha=ha, va=va, fontsize=10, color=INK)

title1 = f"同一人相邻信令中位间隔 {fmt_dur(p50)}，{fmt_pct(share_gt60, 1)} 的间隔超过 1 小时"
subtitle1 = (f"{DAY_LABEL} 共 {n_user:,} 人、{n_gap:,} 个间隔；同一人按时间排序后相邻两条相减；"
             f"\n0 秒间隔（同一时刻重复记录）占 {fmt_pct(share_zero, 1)}，计入曲线起点；"
             f"信令随手机活动产生，间隔长不等于人没动")
assert len(title1) <= 40, len(title1)
viz.finish(fig, title1, subtitle1, SOURCE, "Q01_信令间隔分布.png")

# %% 3 图二：每小时至少有 1 条信令的人数占比（折线）
peak_h = hourly["人数"].idxmax()
low_h = hourly["人数"].idxmin()
pk, lo = hourly.loc[peak_h], hourly.loc[low_h]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hourly.index, hourly["占比"], color=MAIN, lw=2.2, marker="o", ms=3.5)
ax.fill_between(hourly.index, hourly["占比"], 1, color=MAIN_LIGHT, alpha=0.6, lw=0)
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, 1.0)
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f"{h} 点" for h in range(0, 24, 3)])
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xlabel(f"小时（{DAY_LABEL}）")
ax.set_ylabel("该小时至少有 1 条信令的人数占比")
ax.grid(axis="y", color=GRAY, lw=0.6, alpha=0.6)

# 峰值、谷值：打点 + 写人数和占比
for h, row, dy, va in [(peak_h, pk, -12, "top"), (low_h, lo, -12, "top")]:
    ax.plot(h, row["占比"], "o", color=MAIN_DARK, ms=7, zorder=3)
    ax.annotate(f"{h} 点：{row['人数']:,.0f} 人（{fmt_pct(row['占比'])}）", (h, row["占比"]),
                xytext=(0, dy), textcoords="offset points",
                ha="center", va=va, fontsize=10, color=INK)

# 浅色区域 = 当小时看不到的人
ax.text(11.5, 0.97, "浅色区域：该小时没有任何信令、看不到位置的人",
        ha="center", va="top", fontsize=9.5, color=MUTED)

night = "凌晨 " if low_h < 6 else ""
title2 = (f"{peak_h} 点能看到的人最多，占 {fmt_pct(pk['占比'])}；"
          f"{night}{low_h} 点最少，仅 {fmt_pct(lo['占比'])}")
subtitle2 = (f"{DAY_LABEL} 分母为当天出现过的 {n_user:,} 人；某小时内至少有 1 条信令即算“能看到”"
             f"\n没有信令≠不在场：夜间偏低多因手机静默，不能读作人少；此图不是在场人数")
assert len(title2) <= 40, len(title2)
viz.finish(fig, title2, subtitle2, SOURCE, "Q02_每小时可观测人数.png")

plt.show()